In [1]:
import pandas as pd

df = pd.read_csv("../target.csv")
df.head(10)

,itemId,category,name,rating,originalRating,reviewTitle,reviewContent,likeCount,upVotes,downVotes,helpful,relevanceScore,boughtDate,clientType,retrievedDate
0,100002528,beli-harddisk-eksternal,Kamal U.,5,NaN,NaN,bagus mantap dah sesui pesanan,0,0,0,True,26.51,09 Apr 2019,androidApp,2019-10-02
1,100002528,beli-harddisk-eksternal,yofanca m.,4,NaN,NaN,"Bagus, sesuai foto",0,0,0,True,22.49,24 Sep 2017,androidApp,2019-10-02
2,100002528,beli-harddisk-eksternal,Lazada Customer,5,NaN,ok mantaaapppp barang sesuai pesanan.. good,okkkkk mantaaaaaaapppp ... goood,0,0,0,True,21.50,04 Apr 2018,androidApp,2019-10-02
3,100002528,beli-harddisk-eksternal,Lazada Customer,4,NaN,NaN,bagus sesuai,0,0,0,True,20.51,22 Sep 2017,androidApp,2019-10-02
4,100002528,beli-harddisk-eksternal,Yosep M.,5,NaN,NaN,NaN,0,0,0,True,16.01,17 Agu 2018,androidApp,2019-10-02
5,100002528,beli-harddisk-eksternal,Deden,5,NaN,NaN,NaN,0,0,0,True,16.01,02 Nov 2017,androidApp,2019-10-02
6,100002528,beli-harddisk-eksternal,Yeana,5,NaN,NaN,NaN,0,0,0,True,13.01,25 Sep 2017,mobile,2019-10-02
7,100002528,beli-harddisk-eksternal,nurfarida,1,NaN,ada pengirimn ntb bima,bima,4,4,0,True,7.22,NaN,androidApp,2019-10-02
8,100003785,beli-harddisk-eksternal,Fadjar B.,1,NaN,NaN,baru 10 bulan layarnya dah bergaris,0,0,0,True,21.49,06 Apr 2017,androidApp,2019-10-02
9,100003785,beli-harddisk-eksternal,agung p.,5,NaN,Barang bagus sesuai specs,"Pesan rabu sore,minggu sore sampe,,barang sesu...",0,0,0,True,19.50,01 Mar 2017,mobile,2019-10-02


## Text Preprocessing Pipeline

Setup text preprocessing using indoNLP library

In [2]:
from indoNLP.preprocessing import pipeline, emoji_to_words, replace_word_elongation
from indoNLP.preprocessing import SLANG_DATA, SLANG_PATTERN
import re

# Create a safer version of replace_slang that handles missing keys
def safe_replace_slang(text):
    """Replace slang words, but keep original if not found in dictionary"""
    return re.sub(SLANG_PATTERN, lambda mo: SLANG_DATA.get(mo.group(0).lower(), mo.group(0)), text)

# Use safe_replace_slang instead of the library's version
pipe = pipeline([safe_replace_slang, emoji_to_words, replace_word_elongation])

print("✓ Text preprocessing pipeline ready")

✓ Text preprocessing pipeline ready


## Data Cleaning

Remove duplicates and missing values

In [3]:
# Check data before cleaning
print(f"Original data shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

df.dropna(subset=["reviewContent"])

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print(f"\nAfter cleaning:")
print(f"Cleaned data shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

Original data shape: (203787, 15)
Missing values:
itemId                 0
category               0
name                   0
rating                 0
originalRating    203779
reviewTitle       180388
reviewContent      96758
likeCount              0
upVotes                0
downVotes              0
helpful                0
relevanceScore         0
boughtDate          7107
clientType             0
retrievedDate          0
dtype: int64
Duplicate rows: 3113

After cleaning:
Cleaned data shape: (200674, 15)
Missing values: 478846
Duplicate rows: 0


## Create Labels and Final Dataset

Create labels based on score and prepare `target_final.csv`

In [4]:
df['label'] = df['rating'].apply(lambda x: 'positive' if x > 3 else 'negative')

df_final = df[['reviewContent', 'label', 'rating']].copy()

# Clean df_final: remove NA and duplicates
print(f'df_final before cleaning: {len(df_final)} rows')
print(f"Missing values:\n{df_final.isnull().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

df_final = df_final.dropna()
df_final = df_final.drop_duplicates()
df_final = df_final.reset_index(drop=True)

print(f'\ndf_final after cleaning: {len(df_final)} rows')
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

# Display label distribution
print(f'\nLabel distribution:')
print(df_final['label'].value_counts())
print(f'\nFirst 10 rows:')
df_final.head(10)

df_final before cleaning: 200674 rows
Missing values:
reviewContent    93839
label                0
rating               0
dtype: int64
Duplicate rows: 162328

df_final after cleaning: 38341 rows
Missing values: 0
Duplicate rows: 0

Label distribution:
label
positive    33224
negative     5117
Name: count, dtype: int64

First 10 rows:


,reviewContent,label,rating
0,bagus mantap dah sesui pesanan,positive,5
1,"Bagus, sesuai foto",positive,4
2,okkkkk mantaaaaaaapppp ... goood,positive,5
3,bagus sesuai,positive,4
4,bima,negative,1
5,baru 10 bulan layarnya dah bergaris,negative,1
6,"Pesan rabu sore,minggu sore sampe,,barang sesu...",positive,5
7,"Mau tanya ini cicilnya pake apa ya,cc bkn?",negative,1
8,Apakah TV. Tsb. Suda ada anti gores..,positive,5
9,Pengirim barang tidak sesuai janji. Katanya ex...,negative,1


## Apply Text Preprocessing

Clean and normalize text content

In [5]:
print("Applying text preprocessing to 'reviewContent' column...")
print(f"Total rows to process: {len(df_final)}")

# Apply preprocessing pipeline
df_final['reviewContent'] = df_final['reviewContent'].apply(pipe)

print("✓ Text preprocessing complete!")
print("\nSample of preprocessed content:")
df_final.head(10)

Applying text preprocessing to 'reviewContent' column...
Total rows to process: 38341
✓ Text preprocessing complete!

Sample of preprocessed content:


,reviewContent,label,rating
0,bagus mantap deh sesui pesanan,positive,5
1,"Bagus, sesuai foto",positive,4
2,ok mantaaaaaaap ... goood,positive,5
3,bagus sesuai,positive,4
4,bima,negative,1
5,baru 10 bulan layarnya deh bergaris,negative,1
6,"Pesan rabu sore,minggu sore sampai,,barang ses...",positive,5
7,"Mau tanya ini cicilnya pakai apa ya,c bukan?",negative,1
8,Apakah TV. tersebut. sudah ada anti gores..,positive,5
9,Pengirim barang tidak sesuai janji. Katanya ex...,negative,1


In [6]:
df_final.to_csv('../target_final.csv', index=False)
print('Successfully created target_final.csv!')

Successfully created target_final.csv!


## Create Balanced Dataset

Create `target_balanced.csv` with 30,000 positive and 30,000 negative samples

In [7]:
# Check current label distribution
print("Current label distribution:")
print(df_final['label'].value_counts())
print(f"\nTotal samples: {len(df_final)}")

# Sample 30k positive and 30k negative
positive_samples = df_final[df_final['label'] == 'positive'].sample(n=3000, random_state=42)
negative_samples = df_final[df_final['label'] == 'negative'].sample(n=3000, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_samples, negative_samples], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset:")
print(f"Total rows: {len(df_balanced)}")
print(f"\nLabel distribution:")
print(df_balanced['label'].value_counts())
df_balanced.head(10)

Current label distribution:
label
positive    33224
negative     5117
Name: count, dtype: int64

Total samples: 38341

Balanced dataset:
Total rows: 6000

Label distribution:
label
positive    3000
negative    3000
Name: count, dtype: int64


,reviewContent,label,rating
0,kualitas gambar & suaranya mantap..,positive,5
1,flashdisknya payah. buat simpan data enggak bi...,negative,1
2,barang bagus..terima kasih lasada,positive,5
3,Keren Mantab Jos,positive,5
4,"hd rusak enggak bisa dipakai, lebih ditingkatk...",negative,1
5,mantab.. tanpa basa basi langsung order dan 2 ...,positive,5
6,mantap...barang mendarat dengan selamat walau ...,positive,5
7,"Barang orisinal, bisa YouTube, tapi belum bisa...",positive,5
8,harga promo dan kualitas bagus.. thanks lazada...,positive,5
9,good. really nice and fast order,positive,5


In [8]:
# Save balanced dataset to CSV
df_balanced.to_csv('../target_balanced.csv', index=False)